# Sales Forecasting with PyCaret - Exogenous Variables
# Part B: Feature Pre-Processing
## Dr José M Albornoz
### June 2025

In this notebook we will perform the required feature pre-processing to build a sales forecasting model for a brick-and-mortar store.

# 0.- Imports

In [1]:
from pycaret.time_series import *
import pandas as pd
import numpy as np
import plotly.express as px
import datetime
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer

# maximum number of rdataframe ows and columns displayed
pd.set_option('display.max_rows', 50000)
pd.set_option('display.max_columns', 500)

pd.options.mode.chained_assignment = None

RANDOM_SEED = 801

# 1.- Load data

In Part A we encoded text-based features such as **Holiday** and **DestinationEvent** and saved the resulting dataframe. We will now retrieve the encoded dataset.

In [2]:
data = pd.read_csv('../data/sales_with_exogenous_partial_text_encoding.csv')
data.head()

,Date,Sales,Num_Employees,Returns_Pct,Pct_On_Sale,Pct_Promotional,Marketing,Holiday,DestinationEvent,Econ_ChangeGDP,EconJobsChange,AnnualizedCPI
0,7/1/12,638368,33,1.066,10.526,0.000047,July In Store Credit Card Signup Discount; In ...,0,0,0.5,750.0,0.02
1,7/2/12,667057,18,0.423,9.142,0.000047,July In Store Credit Card Signup Discount; In ...,0,0,0.5,750.0,0.02
2,7/3/12,569246,19,0.321,9.466,0.000047,July In Store Credit Card Signup Discount; In ...,0,0,0.5,750.0,0.02
3,7/4/12,599165,27,0.862,10.657,0.000047,July In Store Credit Card Signup Discount; In ...,1,0,0.5,750.0,0.02
4,7/5/12,626678,28,0.523,10.359,0.000047,July In Store Credit Card Signup Discount; ID5...,0,0,0.5,750.0,0.02


In [3]:
data.shape

(714, 12)

In [4]:
data.dtypes

Date                 object
Sales                 int64
Num_Employees         int64
Returns_Pct         float64
Pct_On_Sale         float64
Pct_Promotional     float64
Marketing            object
Holiday               int64
DestinationEvent      int64
Econ_ChangeGDP      float64
EconJobsChange      float64
AnnualizedCPI       float64
dtype: object

In [5]:
type(data.index)

pandas.core.indexes.range.RangeIndex

# 2.- Imputation for macroeconomic indicators

The three macroeconomic indicators included in the dataset (**Econ_ChangeGDP**, **EconJobsChange** and **AnnualizedCPI**) contain missing values as they are generated with quarterly, monthly, and weekly frequencies respectively. We will use forward imputation to replace missing values with the most recent previous value.

In [6]:
data = data.ffill()

In [7]:
data.head()

,Date,Sales,Num_Employees,Returns_Pct,Pct_On_Sale,Pct_Promotional,Marketing,Holiday,DestinationEvent,Econ_ChangeGDP,EconJobsChange,AnnualizedCPI
0,7/1/12,638368,33,1.066,10.526,0.000047,July In Store Credit Card Signup Discount; In ...,0,0,0.5,750.0,0.02
1,7/2/12,667057,18,0.423,9.142,0.000047,July In Store Credit Card Signup Discount; In ...,0,0,0.5,750.0,0.02
2,7/3/12,569246,19,0.321,9.466,0.000047,July In Store Credit Card Signup Discount; In ...,0,0,0.5,750.0,0.02
3,7/4/12,599165,27,0.862,10.657,0.000047,July In Store Credit Card Signup Discount; In ...,1,0,0.5,750.0,0.02
4,7/5/12,626678,28,0.523,10.359,0.000047,July In Store Credit Card Signup Discount; ID5...,0,0,0.5,750.0,0.02


# 3.- Encoding for **Marketing**

In Part A of this notebook we found that **Marketing** contains text describing marketing events happening at the store on a given day, such as *August In Store Credit Card Signup Discount; Back to School Mailer and Billboard Campaign ID4.2822*. 

In this example we will use TF-IDF to encode this feature, but first we will perform some transformations to reduce the dimensionality of the encoding.

## 3.1.- Preliminary treatment

We begin by observing that all values of **Marketing** end with a code with the format *IDxx.xxxx*. We want to know if this code contains any useful signal that could be used for forecasting. For this purpose, we will create a new column containing the code and examine the number of unique values it holds.

In [8]:
# create a new column with marketing id
data['MarketingID'] = data['Marketing'].str.extract(r'(ID\d+\.\d+)')

In [9]:
# number of unique values
data['MarketingID'].nunique()

713

We find that this marketing code has 713 distinct values within a 714 row dataset; we can conclude that the code is only an identifier that contains no useful signal, and can be therefore safely removed from **Marketing**. 

In [10]:
# drop marketing ID from dataframe
data.drop(columns='MarketingID', inplace=True)

In [11]:
# remove identifier from Marketing
regex_pattern_to_remove = r'\s*ID\d+\.\d+'
data['Marketing'] = data['Marketing'].str.replace(regex_pattern_to_remove, '', regex=True)

Through visual inspection we find that all values of **Marketing** contain the same substring: *In Store Credit Card Signup Discount*, and therefore we can remove this substring from the feature to reduce the dimensionality of the subsequent encoding.

In [12]:
# remove substring that is common to all values
substring_to_remove = 'In Store Credit Card Signup Discount'
data['Marketing'] = data['Marketing'].str.replace(substring_to_remove, '', regex=False)

In [13]:
data.head()

,Date,Sales,Num_Employees,Returns_Pct,Pct_On_Sale,Pct_Promotional,Marketing,Holiday,DestinationEvent,Econ_ChangeGDP,EconJobsChange,AnnualizedCPI
0,7/1/12,638368,33,1.066,10.526,0.000047,July ; In store 10% off accessories; Ad Words ...,0,0,0.5,750.0,0.02
1,7/2/12,667057,18,0.423,9.142,0.000047,July ; In store 10% off accessories; Ad Words ...,0,0,0.5,750.0,0.02
2,7/3/12,569246,19,0.321,9.466,0.000047,July ; In store 10% off accessories; Ad Words ...,0,0,0.5,750.0,0.02
3,7/4/12,599165,27,0.862,10.657,0.000047,July ; In store 10% off accessories; Ad Words ...,1,0,0.5,750.0,0.02
4,7/5/12,626678,28,0.523,10.359,0.000047,July ;,0,0,0.5,750.0,0.02


We will also remove punctuation from **Marketing**:

In [14]:
# define the regex pattern for punctuation
# string.punctuation gives you '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'
# re.escape() escapes all special characters in the string.punctuation
# this creates a regex like r'[!"#$%&\'()*+,-./:;<=>?@[\\\]^_`{|}~]'
punctuation_pattern = r'[' + re.escape(string.punctuation) + r']'

# apply str.replace()
data['Marketing'] = data['Marketing'].str.replace(punctuation_pattern, '', regex=True)

In [15]:
data.head()

,Date,Sales,Num_Employees,Returns_Pct,Pct_On_Sale,Pct_Promotional,Marketing,Holiday,DestinationEvent,Econ_ChangeGDP,EconJobsChange,AnnualizedCPI
0,7/1/12,638368,33,1.066,10.526,0.000047,July In store 10 off accessories Ad Words Ind...,0,0,0.5,750.0,0.02
1,7/2/12,667057,18,0.423,9.142,0.000047,July In store 10 off accessories Ad Words Ind...,0,0,0.5,750.0,0.02
2,7/3/12,569246,19,0.321,9.466,0.000047,July In store 10 off accessories Ad Words Ind...,0,0,0.5,750.0,0.02
3,7/4/12,599165,27,0.862,10.657,0.000047,July In store 10 off accessories Ad Words Ind...,1,0,0.5,750.0,0.02
4,7/5/12,626678,28,0.523,10.359,0.000047,July,0,0,0.5,750.0,0.02


In [16]:
data['Marketing'].nunique()

38

After this preliminary treatment he have reduced the number of distinct values in **Marketing** from 713 to 38 - this increases the likelihood of finding useful signal in this feature.

## 3.2.- TF-IDF encoding

We are now ready to encode **Marketing**. To further reduce the dimensionality of the resulting encoding we will generate two encodings for **Marketing**: one using bigrams and another using trigrams. We will then experiment to find which encoding produces the best forecasting results.

### 3.2.1.- Bigram encoding

In [17]:
# initialize TF-IDF Vectorizer
# max_features: limits the number of features (words) to avoid too many columns
# stop_words: removes common words like "the", "a", "is"
# only bigrams
tfidf_vectorizer = TfidfVectorizer(max_features=100, stop_words='english', ngram_range=(2, 2))

In [18]:
# fit and transform the 'Marketing' column
marketing_features_2gram = tfidf_vectorizer.fit_transform(data['Marketing'])

In [19]:
# convert to DataFrame
marketing_features_2gram_df = pd.DataFrame(marketing_features_2gram.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

In [20]:
marketing_features_2gram_df.shape

(714, 72)

In [21]:
# concatenate with original DataFrame
data_2gram = pd.concat([data, marketing_features_2gram_df], axis=1)

In [22]:
# impute all missing values with zeroes
data_2gram.fillna(0, inplace=True)

In [23]:
# drop encoded text feature
data_2gram.drop(columns='Marketing', inplace=True)

In [24]:
data_2gram.head()

,Date,Sales,Num_Employees,Returns_Pct,Pct_On_Sale,Pct_Promotional,Holiday,DestinationEvent,Econ_ChangeGDP,EconJobsChange,AnnualizedCPI,10 accessories,accessories ad,ad words,april ready,april spring,august school,billboard campaign,black friday,blitz tv,campaign fall,campaign labor,campaign memorial,campaign minute,christmas clearance,christmas email,clearance email,columbus day,commercial radio,day email,day mailer,day sale,day sales,dec christmas,door walk,email blast,email campaign,end summer,event campaign,event promotion,fall product,fall summer,friday blitz,gear sale,half email,holiday sales,independence day,july end,july half,july school,july store,july summer,labor day,madness gear,mailer billboard,mailer black,march madness,memorial day,minute holiday,mlk event,october fall,presidents day,product email,product roll,radio spot,ready summer,roll email,sale mailer,sales commercial,sales event,school mailer,september school,spring product,store 10,summer campaign,summer clearance,summer event,summer swim,swim mailer,tv commercial,verterns door,walk sale,words independence
0,7/1/12,638368,33,1.066,10.526,0.000047,0,0,0.5,750.0,0.02,0.353553,0.353553,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553
1,7/2/12,667057,18,0.423,9.142,0.000047,0,0,0.5,750.0,0.02,0.353553,0.353553,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553
2,7/3/12,569246,19,0.321,9.466,0.000047,0,0,0.5,750.0,0.02,0.353553,0.353553,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553
3,7/4/12,599165,27,0.862,10.657,0.000047,1,0,0.5,750.0,0.02,0.353553,0.353553,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.353553
4,7/5/12,626678,28,0.523,10.359,0.000047,0,0,0.5,750.0,0.02,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


In [25]:
# save encoded dataframe
data_2gram.to_csv('../data/data_2gram.csv', index=False)

### 3.2.2.- Trigram encoding

In [26]:
# initialize TF-IDF Vectorizer
# max_features: limits the number of features (words) to avoid too many columns
# stop_words: removes common words like "the", "a", "is"
# only trigrams
tfidf_vectorizer = TfidfVectorizer(max_features=100, stop_words='english', ngram_range=(3, 3))

In [27]:
# fit and transform the 'Marketing' column
marketing_features_3gram = tfidf_vectorizer.fit_transform(data['Marketing'])

In [28]:
# convert to DataFrame
marketing_features_3gram_df = pd.DataFrame(marketing_features_3gram.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

In [29]:
marketing_features_3gram_df.shape

(714, 61)

In [30]:
# concatenate with original DataFrame
data_3gram = pd.concat([data, marketing_features_3gram_df], axis=1)

In [31]:
# impute all missing values with zeroes
data_3gram.fillna(0, inplace=True)

In [32]:
# drop encoded text feature
data_3gram.drop(columns='Marketing', inplace=True)

In [33]:
data_3gram.head()

,Date,Sales,Num_Employees,Returns_Pct,Pct_On_Sale,Pct_Promotional,Holiday,DestinationEvent,Econ_ChangeGDP,EconJobsChange,AnnualizedCPI,10 accessories ad,accessories ad words,ad words independence,april ready summer,april spring product,august school mailer,billboard campaign labor,black friday blitz,blitz tv commercial,campaign fall product,campaign labor day,campaign memorial day,campaign minute holiday,christmas clearance email,christmas email campaign,clearance email campaign,columbus day mailer,commercial radio spot,day email blast,day sales event,dec christmas email,door walk sale,email campaign fall,email campaign minute,end summer event,fall product roll,fall summer clearance,friday blitz tv,holiday sales commercial,independence day sale,july end summer,july half email,july school mailer,july store 10,july summer swim,labor day sales,madness gear sale,mailer billboard campaign,mailer black friday,march madness gear,memorial day email,minute holiday sales,mlk event promotion,october fall product,presidents day email,product roll email,ready summer campaign,roll email campaign,sale mailer black,sales event campaign,school mailer billboard,september school mailer,spring product email,store 10 accessories,summer campaign memorial,summer clearance email,summer swim mailer,tv commercial radio,verterns door walk,walk sale mailer,words independence day
0,7/1/12,638368,33,1.066,10.526,0.000047,0,0,0.5,750.0,0.02,0.377964,0.377964,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.377964
1,7/2/12,667057,18,0.423,9.142,0.000047,0,0,0.5,750.0,0.02,0.377964,0.377964,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.377964
2,7/3/12,569246,19,0.321,9.466,0.000047,0,0,0.5,750.0,0.02,0.377964,0.377964,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.377964
3,7/4/12,599165,27,0.862,10.657,0.000047,1,0,0.5,750.0,0.02,0.377964,0.377964,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.377964,0.0,0.0,0.0,0.0,0.0,0.0,0.377964
4,7/5/12,626678,28,0.523,10.359,0.000047,0,0,0.5,750.0,0.02,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000000


In [34]:
# save encoded dataframe
data_3gram.to_csv('../data/data_3gram.csv', index=False)